# Settimana 3 — L'estate del 1654: contare i futuri possibili

Questo è il notebook della Settimana 3 del corso di Probabilità e Statistica. Contiene gli script Python della [dispensa della settimana](https://qwertyjacob.github.io/PSI-VA-2627/settimana_03/), nello stesso ordine in cui compaiono: ogni script è una cella di codice, preceduta dal titolo della sezione in cui si trova. Le spiegazioni restano nella dispensa.

Per eseguire una cella, premi il pulsante ▶ alla sua sinistra. Gli script sono indipendenti l'uno dall'altro, quindi puoi eseguirli in qualsiasi ordine e modificarli liberamente. Colab non salva le modifiche nel notebook originale: se vuoi conservarle, scegli *File → Salva una copia in Drive*.

## Script 1 — Pascal, Fermat e il Monte Carlo a confronto

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_03/#lab-partis)

In [ ]:
# Script 1 — Il problème des partis: Pascal (ricorsione), Fermat (enumerazione), Monte Carlo
from functools import lru_cache
from itertools import product
import random

@lru_cache(maxsize=None)   # memoizzazione: ogni (a, b, p) viene calcolato una sola volta
def pascal(a, b, p=0.5):
    """P(A vince) quando ad A mancano a punti, a B ne mancano b; A vince ogni manche con prob. p."""
    if a == 0: return 1.0   # A non ha più punti da fare: ha già vinto
    if b == 0: return 0.0   # B ha già vinto
    return p * pascal(a - 1, b, p) + (1 - p) * pascal(a, b - 1, p)   # un'altra manche: due futuri

def fermat(a, b, p=0.5):
    """Si giocano comunque a+b-1 manche (bastano a decidere): si sommano i futuri in cui A ne vince >= a."""
    n = a + b - 1            # numero massimo di manche che servono a decidere
    tot = 0.0
    for futuro in product("AB", repeat=n):   # tutte le 2^n sequenze di vincitori, es. ("A","B","A")
        k = futuro.count("A")                # manche vinte da A in questo futuro
        if k >= a:                           # A raggiunge i punti che gli mancano: A vince
            tot += p**k * (1 - p)**(n - k)   # probabilità di questa precisa sequenza
    return tot

def montecarlo(a, b, p=0.5, prove=100_000, seme=1):
    rng = random.Random(seme)   # generatore con seme fisso: risultato riproducibile
    vinte = 0
    for _ in range(prove):      # simula `prove` partite complete
        x, y = a, b             # punti mancanti ad A e a B
        while x > 0 and y > 0:  # si gioca finché nessuno ha vinto
            if rng.random() < p: x -= 1   # A vince la manche
            else: y -= 1                  # B vince la manche
        vinte += (x == 0)       # A ha vinto la partita (True conta come 1)
    return vinte / prove        # frequenza relativa = stima della probabilità

POSTA = 64
print("Gioco a 3 punti, posta 64 pistole")
for nome, (a, b) in {"2–1": (1, 2), "2–0": (1, 3), "1–0": (2, 3)}.items():
    P = pascal(a, b)   # valore esatto, usato come riferimento
    print(f"  sul {nome}: Pascal {P:.4f} | Fermat {fermat(a, b):.4f} | Monte Carlo {montecarlo(a, b):.4f}"
          f"  →  A: {POSTA * P:.2f}, B: {POSTA * (1 - P):.2f}")
    # controllo: Pascal = Fermat (esatto) e Monte Carlo vicino (tolleranza 0.01)
    assert abs(P - fermat(a, b)) < 1e-12 and abs(P - montecarlo(a, b)) < 0.01

# I numeri della lettera di Pascal del 29 luglio 1654
assert POSTA * pascal(1, 2) == 48 and POSTA * pascal(1, 3) == 56 and POSTA * pascal(2, 3) == 44

# E se le manche non sono ugualmente probabili? Basta cambiare p (qui A vince una manche su 3)
print("Con p = 1/3 per A, sul 2–1:", round(pascal(1, 2, 1/3), 4), "(Fermat:", round(fermat(1, 2, 1/3), 4), ")")
assert abs(pascal(1, 2, 1/3) - fermat(1, 2, 1/3)) < 1e-12
print("Script 1: tutti gli assert superati.")

## Script 2 — Le formule contro l'enumerazione

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_03/#lab-combinatoria)

In [ ]:
# Script 2 — Contare senza elencare: formule, enumerazione e triangolo di Pascal
from math import comb, factorial, perm, log2
from itertools import permutations, combinations, product
import random

# (a) Le formule contro l'enumerazione esplicita (n = 5 chiavi, k = 3 slot)
n, k = 5, 3
assert len(list(product(range(n), repeat=k))) == n**k                       # moltiplicazione: n·n·n
assert len(list(permutations(range(n), k))) == factorial(n) // factorial(n - k) == perm(n, k)   # disposizioni
assert len(list(combinations(range(n), k))) == factorial(n) // (factorial(k) * factorial(n - k)) == comb(n, k)   # combinazioni
assert len(list(permutations(range(n), k))) == len(list(combinations(range(n), k))) * factorial(k)   # ogni sottoinsieme conta k! volte
print(f"n={n}, k={k}: sequenze con ripetizione {n**k}, disposizioni {perm(n, k)}, combinazioni {comb(n, k)}")

# (b) Il triangolo di Pascal con la regola C(n,k) = C(n-1,k-1) + C(n-1,k)
riga_precedente = [1]                           # la riga 0 contiene solo un 1
righe = [riga_precedente]                       # qui conserviamo tutte le righe
for _ in range(10):                             # costruiamo le righe da 1 a 10
    riga_nuova = [1]                            # ogni riga comincia con 1
    for j in range(1, len(riga_precedente)):    # ogni valore interno è la somma dei due valori sopra di lui
        riga_nuova.append(riga_precedente[j - 1] + riga_precedente[j])
    riga_nuova.append(1)                        # e finisce con 1
    righe.append(riga_nuova)
    riga_precedente = riga_nuova                # la riga appena costruita diventa la precedente del prossimo giro

for riga in righe[:7]:              # stampiamo le righe da 0 a 6
    testo = ""
    for valore in riga:
        testo = testo + f"{valore:3d} "
    print(testo.center(40))

# Controlli: ogni casella è C(n,k) = n! / (k! (n-k)!), e ogni riga somma a 2^n
for n in range(11):
    for k in range(n + 1):
        assert righe[n][k] == factorial(n) // (factorial(k) * factorial(n - k))
    assert sum(righe[n]) == 2**n    # i sottoinsiemi di un n-insieme sono 2^n

# (c) Probabilità classica: k teste in n lanci = C(n,k) / 2^n. Esatta contro simulazione
n, k = 10, 5
esatta = comb(n, k) / 2**n
rng = random.Random(7)
prove = 100_000
freq = sum(sum(rng.random() < 0.5 for _ in range(n)) == k for _ in range(prove)) / prove
print(f"P({k} teste su {n}): esatta {esatta:.4f}, simulata {freq:.4f}")
assert abs(esatta - freq) < 0.006

# (d) Gli ordinamenti di n elementi sono n!: servono almeno log2(n!) confronti per distinguerli
print("log2(100!) =", round(log2(factorial(100)), 1), "confronti almeno, per ordinare 100 elementi")
print("Script 2: tutti gli assert superati.")

## Script 3 — De Méré, esatto e simulato

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_03/#lab-de-mere)

In [ ]:
# Script 3 — Il «paradosso» di de Méré: esatto (complementare) e simulato
import numpy as np

def almeno_un_successo(p, tentativi):
    """P(almeno un successo in `tentativi` prove indipendenti): 1 - P(nessun successo)."""
    return 1 - (1 - p) ** tentativi

un_sei    = almeno_un_successo(1 / 6, 4)        # un dado, 4 lanci
doppio_sei = almeno_un_successo(1 / 36, 24)     # due dadi, 24 lanci
print(f"Almeno un 6 in 4 lanci:           {un_sei:.4f}  (= 671/1296 = {671/1296:.4f})")
print(f"Almeno un doppio 6 in 24 lanci:   {doppio_sei:.4f}")
assert abs(un_sei - 671 / 1296) < 1e-12 and un_sei > 0.5 > doppio_sei

# Il ragionamento «per proporzione» somma le probabilità: n·p. Ma P(almeno uno) < n·p
print(f"n·p vale {4/6:.4f} e {24/36:.4f}: stesso numero, ma le probabilità vere sono diverse")

# Quanti lanci di due dadi servono per superare 1/2?
n = 1
while almeno_un_successo(1 / 36, n) <= 0.5:
    n += 1
print(f"Con 24 lanci: {doppio_sei:.4f}; servono {n} lanci per arrivare a {almeno_un_successo(1/36, n):.4f}")
assert n == 25

# Verifica Monte Carlo: simuliamo T giocate complete e contiamo la frazione "vincente"
rng = np.random.default_rng(2024)   # seed fisso: risultati riproducibili
T = 100_000                         # numero di giocate simulate
# Gioco 1: ogni riga è una giocata da 4 lanci di un dado (valori 1..6)
d1 = rng.integers(1, 7, size=(T, 4), dtype=np.int8)
# any(axis=1): la giocata vince se ALMENO un lancio è 6; mean() = frequenza relativa
mc1 = (d1 == 6).any(axis=1).mean()
# Gioco 2: T giocate x 24 lanci x 2 dadi (ultimo asse = i due dadi del lancio)
d2 = rng.integers(1, 7, size=(T, 24, 2), dtype=np.int8)
# d2[g, l, k] = esito del dado k (0 o 1) nel lancio l della giocata g
# Passo 1: il primo dado è 6? Matrice di True/False di forma (T, 24)
primo_e_sei = (d2[:, :, 0] == 6)
# Passo 2: il secondo dado è 6? Stessa forma (T, 24)
secondo_e_sei = (d2[:, :, 1] == 6)
# Passo 3: "doppio 6" = entrambi 6 nello stesso lancio (& agisce elemento per elemento)
doppio_sei_lancio = primo_e_sei & secondo_e_sei      # forma (T, 24)
# Passo 4: la giocata vince se ALMENO un lancio dei 24 è doppio 6.
# any(axis=1) collassa l'asse dei lanci: da (T, 24) a (T,), un True/False per giocata
giocata_vinta = doppio_sei_lancio.any(axis=1)
# Passo 5: la media di valori booleani è la frazione di True (True=1, False=0),
# cioè la frequenza relativa delle giocate vinte: la stima Monte Carlo
mc2 = giocata_vinta.mean()
print(f"Monte Carlo ({T} giocate): un 6 in 4 lanci {mc1:.4f}, doppio 6 in 24 lanci {mc2:.4f}")
# Le frequenze simulate devono avvicinarsi alle probabilità esatte (tolleranza 0.01)
assert abs(mc1 - un_sei) < 0.01 and abs(mc2 - doppio_sei) < 0.01
print("Script 3: tutti gli assert superati.")

## Script 4 — Compleanni: esatto, approssimato, simulato

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_03/#lab-compleanni)

In [ ]:
# Script 4 — Compleanni: formula esatta, approssimazione esponenziale e simulazione
from math import exp, sqrt
import random

def p_coincidenza(n, N=365):
    """Esatta: 1 - prod_{i=0}^{n-1} (1 - i/N)  (definizione classica + complementare)."""
    p_distinti = 1.0
    for i in range(n):
        p_distinti *= (N - i) / N
    return 1 - p_distinti

def approssimata(n, N=365):
    """1 - exp(-n(n-1)/2N): da 1 - x <= e^{-x}."""
    return 1 - exp(-n * (n - 1) / (2 * N))

def simula(n, N=365, prove=20_000, seme=3):
    """Stima la probabilità di coincidenza ripetendo l'esperimento `prove` volte."""
    rng = random.Random(seme)           # generatore casuale con seme fisso: risultati riproducibili
    con_coincidenza = 0                 # in quante prove almeno due persone hanno lo stesso giorno

    for _ in range(prove):
        # Una prova: assegniamo a ognuna delle n persone un giorno a caso tra 0 e N-1.
        giorni = [rng.randrange(N) for _ in range(n)]

        # Se tutti i giorni sono diversi, l'insieme (set) li tiene tutti: ha n elementi.
        # Se ce n'è qualcuno ripetuto, l'insieme ne ha meno di n.
        if len(set(giorni)) < n:
            con_coincidenza += 1

    return con_coincidenza / prove      # frequenza relativa = stima della probabilità

print(" n   esatta  approx.  simulata")
for n in (10, 20, 23, 30, 40, 50, 57, 70):
    e, a, s = p_coincidenza(n), approssimata(n), simula(n)
    print(f"{n:3d}  {e:.4f}  {a:.4f}   {s:.4f}")
    assert a <= e + 1e-12                      # 1 - x <= e^{-x}: l'approssimazione è per difetto
    assert abs(e - s) < 0.015

assert round(p_coincidenza(23), 4) == 0.5073 and round(p_coincidenza(22), 4) == 0.4757

# La soglia del 50%: il più piccolo n con probabilità >= 1/2, contro la regola sqrt(2 ln2 · N)
def soglia(N):
    n = 1
    while p_coincidenza(n, N) < 0.5:
        n += 1
    return n
for N in (365, 10_000, 1_000_000):
    print(f"N = {N:>9,}:  n_50% = {soglia(N):>5}   regola 1.1774·√N = {1.1774 * sqrt(N):8.1f}")
assert soglia(365) == 23 and abs(soglia(1_000_000) - 1.1774 * 1000) < 2

# «Qualcuno è nato il MIO compleanno»: tutt'altra probabilità
print(f"Con 22 altre persone, P(qualcuno condivide il mio compleanno) = {1 - (364/365)**22:.4f}")
print("Script 4: tutti gli assert superati.")

## Script 5 — Il birthday attack su un hash troncato

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_03/#lab-hash)

In [ ]:
# Script 5 — Birthday attack: prima collisione di un SHA-256 troncato
import hashlib
import random
from statistics import median


def prima_collisione(b, rng):
    """Hasha messaggi casuali, tenendo solo i primi b bit dell'hash,
    finché due messaggi hanno lo stesso valore. Restituisce quanti ne
    sono serviti."""
    visti = set()   # hash troncati già incontrati (controllo veloce)
    n = 0           # quanti messaggi abbiamo hashato finora
    while True:
        # messaggio casuale di 8 byte (64 bit): quasi sempre nuovo
        # ("big" = big-endian: byte più significativo per primo)
        msg = rng.getrandbits(64).to_bytes(8, "big")
        # SHA-256 dà 256 bit: ">> (256 - b)" tiene solo i primi b
        # (lo shift a destra scarta gli ultimi 256 - b bit)
        digest = hashlib.sha256(msg).digest()
        h = int.from_bytes(digest, "big") >> (256 - b)
        n += 1
        if h in visti:  # stesso hash di un messaggio precedente
            return n
        visti.add(h)


rng = random.Random(2026)   # seme fisso: risultati sempre uguali
print("bit   prove   mediana   1,18·2^(b/2)   prove con n <= soglia")
# per ogni b, quante prove ripetere (più b è grande, più ogni prova
# costa, quindi meno prove)
for b, prove in ((12, 400), (16, 300), (20, 100), (24, 30)):
    # numero di messaggi dopo cui la collisione ha il 50% di probabilità
    soglia = 1.18 * 2 ** (b / 2)
    # ripetiamo l'esperimento: un risultato per ogni prova
    risultati = [prima_collisione(b, rng) for _ in range(prove)]
    sotto = sum(n <= soglia for n in risultati)
    percento = 100 * sotto / prove
    print(f"{b:3d}  {prove:>6}  {median(risultati):>8.0f}  "
          f"{soglia:>13.0f}  {percento:>20.0f}%")
    # la mediana misurata deve stare entro il 30% della soglia teorica
    assert 0.7 * soglia < median(risultati) < 1.3 * soglia
    # e circa metà delle prove deve finire entro la soglia
    assert 30 <= percento <= 70
print("Script 5: tutti gli assert superati.")